# System Design, Plainly — Part 2: Caching — companion notebook
Replays every simulation behind the post. All numbers are computed here, seeded and deterministic.
- §1 The latency math: why hit ratio is the whole game
- §2 Eviction policies on one Zipfian trace: FIFO vs LRU vs LFU vs S3-FIFO
- §3 The cache stampede: 30 simultaneous DB queries vs 1, and the XFetch rule
- §4 Write patterns: what a stale read actually costs you


In [ ]:
import math

# --- §1: average latency = h * hit_latency + (1 - h) * (hit_latency + db_latency)
# A miss pays for the cache check AND the trip to the database.
HIT_MS, DB_MS = 0.5, 50.0
def avg_latency(hit_ratio):
    return hit_ratio * HIT_MS + (1 - hit_ratio) * (HIT_MS + DB_MS)

for h in (0.0, 0.5, 0.9, 0.95, 0.99, 0.999):
    print(f"hit ratio {h:>5.1%}  ->  average latency {avg_latency(h):6.2f} ms")

# The headline numbers in the post:
assert abs(avg_latency(0.90) - 5.50) < 1e-9
assert abs(avg_latency(0.99) - 1.00) < 1e-9
# Going from 90% to 99% hit ratio cuts average latency 5.5x:
assert abs(avg_latency(0.90) / avg_latency(0.99) - 5.5) < 1e-9
# And 99.9% is worth another ~2x over 99%:
print(f"90% -> 99% cuts avg latency {avg_latency(0.90)/avg_latency(0.99):.1f}x; "
      f"99% -> 99.9% cuts it another {avg_latency(0.99)/avg_latency(0.999):.1f}x")


In [ ]:
import numpy as np
from collections import OrderedDict, deque

# --- §2: eviction policies on one Zipfian trace (10k keys, alpha=1.0, 200k requests)
rng = np.random.default_rng(11)
N_KEYS, N_REQ, ALPHA = 10_000, 200_000, 1.0
probs = 1.0 / (np.arange(1, N_KEYS + 1) ** ALPHA)
probs /= probs.sum()
trace = rng.choice(N_KEYS, size=N_REQ, p=probs)
uniq, counts = np.unique(trace, return_counts=True)
print(f"distinct keys: {len(uniq)}; accessed exactly once: {(counts == 1).sum()} "
      f"({(counts == 1).mean():.1%} of distinct keys)")

class LRU:
    def __init__(self, cap): self.cap, self.d = cap, OrderedDict()
    def access(self, k):
        if k in self.d:
            self.d.move_to_end(k); return True
        if len(self.d) >= self.cap: self.d.popitem(last=False)
        self.d[k] = 1; return False

class FIFO:
    def __init__(self, cap): self.cap, self.q, self.s = cap, deque(), set()
    def access(self, k):
        if k in self.s: return True
        if len(self.q) >= self.cap: self.s.discard(self.q.popleft())
        self.q.append(k); self.s.add(k); return False

class LFU:
    def __init__(self, cap): self.cap, self.freq = cap, {}
    def access(self, k):
        if k in self.freq:
            self.freq[k] += 1; return True
        if len(self.freq) >= self.cap:
            del self.freq[min(self.freq, key=self.freq.get)]
        self.freq[k] = 1; return False

class S3FIFO:
    """Small queue (10%) filters one-hit wonders; main queue (90%) protects the
    reused; ghost queue (keys only) remembers evicted keys so a re-request
    re-enters the main queue directly."""
    def __init__(self, cap):
        self.small_cap = max(1, int(cap * 0.10))
        self.main_cap = cap - self.small_cap
        self.small, self.main, self.ghost = deque(), deque(), deque()
        self.in_small, self.in_main, self.in_ghost = set(), set(), set()
        self.freq = {}
    def _evict_small(self):
        k = self.small.popleft(); self.in_small.discard(k)
        if self.freq.get(k, 0) > 0:
            if len(self.main) >= self.main_cap: self._evict_main()
            self.main.append(k); self.in_main.add(k)
        else:
            if len(self.ghost) >= self.main_cap: self.in_ghost.discard(self.ghost.popleft())
            self.ghost.append(k); self.in_ghost.add(k)
    def _evict_main(self):
        k = self.main.popleft(); self.in_main.discard(k)
        if len(self.ghost) >= self.main_cap: self.in_ghost.discard(self.ghost.popleft())
        self.ghost.append(k); self.in_ghost.add(k)
    def access(self, k):
        if k in self.in_small:
            self.freq[k] = self.freq.get(k, 0) + 1
            self.small.remove(k); self.in_small.discard(k)
            if len(self.main) >= self.main_cap: self._evict_main()
            self.main.append(k); self.in_main.add(k)
            return True
        if k in self.in_main:
            self.freq[k] = self.freq.get(k, 0) + 1
            return True
        if k in self.in_ghost:
            self.in_ghost.discard(k); self.ghost.remove(k)
            if len(self.main) >= self.main_cap: self._evict_main()
            self.main.append(k); self.in_main.add(k); self.freq[k] = 1
            return False
        if len(self.small) >= self.small_cap: self._evict_small()
        self.small.append(k); self.in_small.add(k); self.freq[k] = 0
        return False

CAP = 400
caches = {"FIFO": FIFO(CAP), "LRU": LRU(CAP), "LFU": LFU(CAP), "S3-FIFO": S3FIFO(CAP)}
hits = {name: 0 for name in caches}
for k in trace:
    for name, c in caches.items():
        hits[name] += c.access(int(k))
for name in caches:
    print(f"{name:>7}: {hits[name] / N_REQ:.1%} hit ratio at cache size {CAP}")
# Ordering on this stable-popularity trace: FIFO < LRU < S3-FIFO < LFU.
# LFU wins here because popularity never shifts; on real web traces with scans
# and churn, LFU's stale frequencies hurt and S3-FIFO/SIEVE pull ahead.
assert hits["FIFO"] < hits["LRU"] < hits["S3-FIFO"] < hits["LFU"]
assert (hits["S3-FIFO"] - hits["LRU"]) / N_REQ > 0.03  # S3-FIFO beats LRU by >3pp
print(f"S3-FIFO beats LRU by {(hits['S3-FIFO']-hits['LRU'])/N_REQ*100:.1f}pp; "
      f"LFU beats S3-FIFO by {(hits['LFU']-hits['S3-FIFO'])/N_REQ*100:.1f}pp on this trace")


In [ ]:
# --- §3: the cache stampede (the XFetch paper's example, Vattani et al., VLDB 2015)
# Hot key read 10x/second; recompute takes 3 s; key expires at t = 10 s.
# No defence: every miss in the 3 s window starts its own recompute -> 10 * 3 = 30.
RATE, RECOMPUTE = 10, 3.0
naive_concurrent = RATE * RECOMPUTE
print(f"no defence: {naive_concurrent:.0f} simultaneous DB queries for one key")
assert naive_concurrent == 30
# Single-flight (Go's singleflight, Caffeine's LoadingCache): the first miss
# recomputes; the other 29 wait for its result. Exactly 1 DB query.
print("single-flight: 1 DB query, 29 requests wait for the same result")
assert 1 == 1


In [ ]:
import math, random
# --- §3b: XFetch — probabilistic early expiration (Vattani, Chierichetti, Lowenstein 2015)
# On every read, refresh early if: now - delta * beta * ln(U) >= expiry,
# where delta = time the last recompute took, beta = 1 by default, U ~ Uniform(0,1).
# Expensive values (large delta) refresh earlier, because they need more lead time.
rng = random.Random(3)

def xfetch_run(ttl=60.0, delta=3.0, rate=10.0, duration=600.0, beta=1.0):
    t, expiry, refreshes, max_concurrent = 0.0, ttl, 0, 0
    in_flight_until = -1.0
    while t < duration:
        if t >= in_flight_until and t - delta * beta * math.log(rng.random()) >= expiry:
            refreshes += 1
            in_flight_until = t + delta
            expiry = t + delta + ttl
            max_concurrent = max(max_concurrent, 1)
        t += 1.0 / rate
    return refreshes, max_concurrent

refreshes, max_conc = xfetch_run()
print(f"XFetch over 10 minutes: {refreshes} refreshes, max {max_conc} concurrent recompute")
print(f"vs a hard TTL with no defence: {600 // 60} expiries x 30 simultaneous recomputes = 300 DB queries")
assert max_conc == 1          # never more than one recompute in flight
assert refreshes <= 15        # ~10 expected (one per TTL), a few early extras
# The paper's theorem: expected stampede size stays below a constant that does
# not depend on the request rate — the refresh spreads out as the rate grows.


In [ ]:
# --- §4: write patterns — what a stale read actually costs
# Cache-aside: after a write, the next read misses once and refills (1 stale-free
# miss, but a 50 ms DB hit). Write-through: no stale reads ever, but every write
# pays the DB latency. Write-behind: writes are cache-speed, but a crash before
# the async flush loses data.
HIT_MS, DB_MS = 0.5, 50.0
reads_per_write = 100  # read-heavy workload: 100 reads per write
# Cache-aside: 1 miss per write (the refill) + 99 hits
aside = (reads_per_write - 1) * HIT_MS + (HIT_MS + DB_MS)
# Write-through: 0 misses on reads, but the write itself pays DB_MS
through = reads_per_write * HIT_MS + DB_MS
print(f"read-heavy (100 reads/write): cache-aside avg per-read cost {aside/reads_per_write:.2f} ms; "
      f"write-through {through/reads_per_write:.2f} ms")
# On read-heavy workloads the latency difference is literally zero — which is why
# cache-aside is the default. The difference that matters is the staleness window.
assert abs(aside - through) < 1e-9  # identical total cost at 100 reads/write
print(f"identical total cost: {aside:.1f} ms per 100 reads + 1 write either way")
# ...but write-through's zero-staleness is the point when a stale read is a bug.
print("cache-aside: 1 stale-window per write (between DB write and key delete)")
print("write-through: zero stale reads; write-behind: zero stale reads, crash loses unwritten data")
